# Sampling SFT — saved workflows

Tickets 01–10 · optional CLI client. Preparation, review and reporting run on CPU.
Real model work uses separately approved RunPod jobs.


## 1. Select the checkout and artifact root

Use the existing artifact tree. For Colab CPU, mount Drive first; set `REPO_DIR` to your checked-out release.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / "pilot_eval").is_dir():
    REPO_DIR = REPO_DIR.parent
if not (REPO_DIR / "pilot_eval").is_dir():
    raise RuntimeError("Set REPO_DIR to the checked-out spar-pilot repository.")

SOURCE_RELEASE = "173b120337b9301f45a96cf3564634e9df54a983"  # Verified execution-code release.
if SOURCE_RELEASE:
    release_check = subprocess.run(
        [sys.executable, "-m", "pilot_eval", "sampling-sft-handoff-release", "--commit", SOURCE_RELEASE],
        cwd=REPO_DIR, text=True, capture_output=True,
    )
    if release_check.returncode:
        raise RuntimeError(release_check.stderr.strip())
    print("Verified source release:", json.loads(release_check.stdout)["release_commit"])

ARTIFACT_ROOT = REPO_DIR / "artifacts"
PLAN_NAME = "sampling-sft-v1"
# In Colab CPU, uncomment before setting ARTIFACT_ROOT:
# from google.colab import drive
# drive.mount("/content/drive")
# ARTIFACT_ROOT = Path("/content/drive/MyDrive/SPAR/spar-pilot/artifacts")


## 2. Select the existing source configurations

Gold may be its original `sft.config.json` or the prepared SFT activation configuration that explicitly points to it.
GRPO uses its existing `activation.prepared.json`. Paths are relative to the artifact root; no directories are guessed from evaluation reports.


In [ ]:
SFT_SOURCE = "plans/pilot3-inputs-v1/activation.prepared.json"
GRPO_SOURCE = "plans/pilot4-grpo-fp32-v1-writes-v1/activation.prepared.json"

selection_path = ARTIFACT_ROOT / "plans" / PLAN_NAME / "source-selection.json"
selection = {"sft": SFT_SOURCE, "grpo": GRPO_SOURCE}
selection_path.parent.mkdir(parents=True, exist_ok=True)
if selection_path.exists() and json.loads(selection_path.read_text()) != selection:
    raise RuntimeError("Saved selection differs. Use a new PLAN_NAME to preserve the old plan.")
if not selection_path.exists():
    selection_path.write_text(json.dumps(selection, indent=2) + "\n")

def run_command(operation, *arguments):
    return subprocess.run(
        [sys.executable, "-u", "-m", "pilot_eval", operation,
         "--output-root", str(ARTIFACT_ROOT), *map(str, arguments)],
        cwd=REPO_DIR, check=True,
    )


## 3. Prepare the immutable plan

Saves the selected paths and approved recipe. Existing compatible plans are reused.


In [ ]:
run_command("sampling-sft-prepare", "--selection", selection_path, "--name", PLAN_NAME)
PLAN_PATH = ARTIFACT_ROOT / "plans" / PLAN_NAME / "sampling-sft.plan.json"


## 4. Audit the original sources

Checks the 512 training examples, 150 held-out identities, saved masks/order, original L4 runtime, training completion and all five adapter checkpoints.
Raw historical files are read only. Output is `source-audited` or `blocked`; neither means engine qualification or experiment execution.


In [ ]:
run_command("sampling-sft-audit", "--config", PLAN_PATH)


## 5. Verify and restore

After a reconnect, run cells 1–2, then this cell. The saved plan determines the paths; no generation is restarted.
If source contents or availability changed, preserve this plan and prepare a new named one.


In [ ]:
PLAN_PATH = ARTIFACT_ROOT / "plans" / PLAN_NAME / "sampling-sft.plan.json"
run_command("sampling-sft-status", "--config", PLAN_PATH)
run_command("sampling-sft-verify", "--config", PLAN_PATH)

config = json.loads(PLAN_PATH.read_text())
audit_path = ARTIFACT_ROOT / config["run_path"] / "results" / "source-audit.json"
audit = json.loads(audit_path.read_text())
print("Audit:", audit_path)
print("Sources ready:", audit["source_ready"])
print("Next:", "engine qualification (not implemented yet)" if audit["source_ready"] else "resolve the listed source blockers in a new plan")


## Current boundary

Implemented: sources, lifecycle, sampler/pilot checks, generation, eligibility review, data gate, training export/readiness and the selected endpoint runner.
Real collection, full pinned-trainer T16, paid approvals and L4 training execution remain separate. Optional stages below are disabled until their saved inputs are selected.


## 6. Inspect the shared budget (CPU)

Initializes or restores the one experiment ledger. It reserves no money and starts no job.
The CLI/run-script guide is `docs/operations/sampling-sft-runpod.md`; this notebook is optional.


In [ ]:
run_command("sampling-sft-budget-init", "--config", PLAN_PATH)
run_command("sampling-sft-budget-status")


## 7. Optional offline lifecycle smoke

Disabled by default. Uses a built-in no-op and simulated local copies, with zero cloud cost.
It cannot qualify RunPod, issue paid approval, or stop a real Pod.


In [ ]:
RUN_OFFLINE_LIFECYCLE_SMOKE = False
if RUN_OFFLINE_LIFECYCLE_SMOKE:
    request = {
        "job_id": "lifecycle-demo", "kind": "noop", "stage": "sampling",
        "mode": "noop", "command": [],
        "outputs": [f"runs/sampling-sft/{PLAN_NAME}/lifecycle-demo/result.json"],
    }
    request_path = PLAN_PATH.parent / "demo-request.json"
    request_path.write_text(json.dumps(request, indent=2) + "\n")
    job_dir = ARTIFACT_ROOT / "runs" / "sampling-sft" / PLAN_NAME / "jobs" / "lifecycle-demo"
    job_path = job_dir / "job.plan.json"
    export_dir = ARTIFACT_ROOT / "exports" / PLAN_NAME / "lifecycle-demo"
    run_command("sampling-sft-job-prepare", "--config", PLAN_PATH, "--request", request_path)
    run_command("sampling-sft-job-run", "--job", job_path)
    for role in ("network", "downloaded"):
        run_command("sampling-sft-job-export", "--job", job_path,
                    "--copy-root", export_dir / role, "--role", role,
                    "--receipt", export_dir / f"{role}-receipt.json")
    run_command("sampling-sft-job-finish", "--job", job_path,
                "--network-receipt", export_dir / "network-receipt.json",
                "--download-receipt", export_dir / "downloaded-receipt.json")


## 8. Optional bounded engine plan (CPU)

Supply a reviewed settings file from the engine runbook. This freezes inputs; it starts no model.
Run the resulting plan only through an approved budgeted job, using the CLI run script.


In [ ]:
ENGINE_SETTINGS = ""  # Existing explicit settings JSON; leave empty to skip.
ENGINE_NAME = PLAN_NAME + "-engine-bounded-v1"
if ENGINE_SETTINGS:
    run_command("sampling-sft-instrument-prepare", "--config", PLAN_PATH,
                "--settings", Path(ENGINE_SETTINGS), "--name", ENGINE_NAME)
    ENGINE_PLAN = ARTIFACT_ROOT / "plans" / ENGINE_NAME / "sampling-sft.instrument.json"
    run_command("sampling-sft-instrument-status", "--config", ENGINE_PLAN)
else:
    print("Engine plan skipped. CLI/run-script instructions: docs/operations/sampling-sft-engine.md")


## 9. Optional pilot plan (CPU)

Supply the selected Stage0.5 instrument plan. This prepares the fixed32 k5 pilot and reads saved progress.
GPU jobs, controlled interruption and recovery use `docs/operations/sampling-sft-pilot.md`.


In [ ]:
SELECTED_INSTRUMENT = ""  # Existing selected instrument config; leave empty to skip.
PILOT_NAME = PLAN_NAME + "-pilot-v1"
if SELECTED_INSTRUMENT:
    run_command("sampling-sft-pilot-prepare", "--instrument", SELECTED_INSTRUMENT,
                "--name", PILOT_NAME)
    PILOT_PLAN = ARTIFACT_ROOT / "plans" / PILOT_NAME / "sampling-sft.pilot.json"
    run_command("sampling-sft-pilot-status", "--config", PILOT_PLAN)
else:
    print("Pilot plan skipped. No GPU job started.")


## 10. Optional generation plan (CPU)

Select the existing pilot configuration. This freezes all512 problems for k0/k2/k5 and reads saved progress.
Stage2 review and paid jobs follow `docs/operations/sampling-sft-generation.md`.


In [ ]:
GENERATION_PILOT = ""  # Existing pilot configuration; leave empty to skip.
GENERATION_NAME = PLAN_NAME + "-generation-v1"
if GENERATION_PILOT:
    run_command("sampling-sft-generation-prepare", "--pilot", GENERATION_PILOT,
                "--name", GENERATION_NAME)
    GENERATION_PLAN = ARTIFACT_ROOT / "plans" / GENERATION_NAME / "sampling-sft.generation.json"
    run_command("sampling-sft-generation-status", "--config", GENERATION_PLAN)
else:
    print("Generation plan skipped. No model or paid job started.")


## 11. Optional eligibility review (CPU)

Select a completed three-arm generation configuration. This creates the source-bound review page and shows required work.
Open the printed page path, identify intended answers, then download a draft. CLI instructions: `docs/operations/sampling-sft-review.md`.


In [ ]:
REVIEW_GENERATION = ""  # Existing completed generation configuration; empty skips.
REVIEW_NAME = PLAN_NAME + "-eligibility-v1"
if REVIEW_GENERATION:
    run_command("sampling-sft-review-prepare", "--generation", REVIEW_GENERATION,
                "--name", REVIEW_NAME)
    REVIEW_PLAN = ARTIFACT_ROOT / "plans" / REVIEW_NAME / "sampling-sft.review.json"
    run_command("sampling-sft-review-status", "--config", REVIEW_PLAN)
else:
    print("Review preparation skipped.")


## 12. Save a human draft or freeze coverage (CPU)

Both actions are disabled by default. Confirm only labels you have personally reviewed.
After import, export to a new filename with `sampling-sft-review-export` and import that saved revision into the page before further edits.
Freeze requires all required human labels; coverage does not authorize training.


In [ ]:
REVIEW_CONFIG = ""  # Saved sampling-sft.review.json.
REVIEW_LABELS = ""  # Downloaded draft JSON.
CONFIRM_HUMAN_IMPORT = False
FREEZE_REVIEWED_COVERAGE = False
if CONFIRM_HUMAN_IMPORT:
    assert REVIEW_CONFIG and REVIEW_LABELS, "Select the review config and downloaded draft."
    run_command("sampling-sft-review-import", "--config", REVIEW_CONFIG,
                "--labels", REVIEW_LABELS, "--origin", "human", "--confirm-human")
if FREEZE_REVIEWED_COVERAGE:
    assert REVIEW_CONFIG, "Select the saved review config."
    run_command("sampling-sft-review-freeze", "--config", REVIEW_CONFIG)
    run_command("sampling-sft-review-verify", "--config", REVIEW_CONFIG)


## 13. Optional gate plan (CPU)

Choose frozen reviewed coverage and the approved word-policy JSON. This prepares score units; it loads no model.
Collect HF FP32 scores through the reserved CLI job route in `docs/operations/sampling-sft-gate.md`.


In [ ]:
GATE_REVIEW = ""  # Saved sampling-sft.review.json.
GATE_WORD_POLICY = ""  # Approved unicode-words-v1 policy JSON.
GATE_NAME = PLAN_NAME + "-gate-v1"
if GATE_REVIEW and GATE_WORD_POLICY:
    run_command("sampling-sft-gate-prepare", "--review", GATE_REVIEW,
                "--word-policy", GATE_WORD_POLICY, "--name", GATE_NAME)
    GATE_PLAN = ARTIFACT_ROOT / "plans" / GATE_NAME / "sampling-sft.gate.json"
    run_command("sampling-sft-gate-status", "--config", GATE_PLAN)
else:
    print("Gate preparation skipped. No model or paid job started.")


## 14. Optional gate report (CPU)

After all independent HF score units are sealed, build and verify the report. This does not authorize training.
Use the CLI gate-request/gate-approve workflow to record an explicit arm selection and current budget review.


In [ ]:
GATE_CONFIG = ""  # Existing gate config with completed HF score evidence.
RUN_GATE_REPORT = False
if RUN_GATE_REPORT:
    assert GATE_CONFIG, "Select the saved gate configuration."
    run_command("sampling-sft-gate-report", "--config", GATE_CONFIG)
    run_command("sampling-sft-gate-verify", "--config", GATE_CONFIG)


## 15. Optional training-data export (CPU)

Select frozen reviewed coverage. Exports stored IDs, original step slots and exact gold checkpoint 0; no training starts.


In [ ]:
TRAINING_REVIEW = ""  # Existing frozen sampling-sft.review.json.
TRAINING_DATA_NAME = PLAN_NAME + "-training-data-v1"
if TRAINING_REVIEW:
    run_command("sampling-sft-training-data-prepare", "--review", TRAINING_REVIEW, "--name", TRAINING_DATA_NAME)
    TRAINING_DATA_CONFIG = ARTIFACT_ROOT / "plans" / TRAINING_DATA_NAME / "sampling-sft.training-data.json"
    run_command("sampling-sft-training-data-export", "--config", TRAINING_DATA_CONFIG)
    run_command("sampling-sft-training-data-verify", "--config", TRAINING_DATA_CONFIG)


## 16. Optional short-batch readiness check (CPU)

Requires an existing pinned trainer environment. Runs tiny offline batches 3–7; no model download or full-arm training. Missing checks remain pending. See `docs/operations/sampling-sft-training-data.md`.


In [ ]:
TRAINING_DATA_CONFIG = ""  # Existing sampling-sft.training-data.json.
READINESS_PYTHON = sys.executable
RUN_FULL_T16 = False
if TRAINING_DATA_CONFIG:
    if RUN_FULL_T16:
        run_command("sampling-sft-training-data-check", "--config", TRAINING_DATA_CONFIG, "--mode", "full", "--python", READINESS_PYTHON)
    run_command("sampling-sft-training-data-status", "--config", TRAINING_DATA_CONFIG)


## 17. Optional endpoint training plan (CPU)

Select the exact export and post-gate selection receipt. This prepares one k0 or k5 attempt; it does not launch training. Full T16 and fresh paid-job approval are required separately.


In [ ]:
ENDPOINT_DATA = ""  # Saved sampling-sft.training-data.json.
ENDPOINT_SELECTION = ""  # Saved training-selections/<hash>.json.
ENDPOINT_K = 0
ENDPOINT_NAME = PLAN_NAME + f"-k{ENDPOINT_K}-attempt1"
if ENDPOINT_DATA and ENDPOINT_SELECTION:
    run_command("sampling-sft-train-prepare", "--data", ENDPOINT_DATA, "--selection", ENDPOINT_SELECTION, "--k", ENDPOINT_K, "--name", ENDPOINT_NAME)
    ENDPOINT_CONFIG = ARTIFACT_ROOT / "plans" / ENDPOINT_NAME / "sampling-sft.training.json"
    run_command("sampling-sft-train-status", "--config", ENDPOINT_CONFIG)


## 18. Optional training job request (CPU)

Supply an explicit rate/time/volume template. This writes child argv and required exports; no reservation or job is launched here. Use the CLI lifecycle runbook for approval, execution, export and shutdown.


In [ ]:
ENDPOINT_CONFIG = globals().get("ENDPOINT_CONFIG", "")  # Restore section 17 or select a saved plan.
TRAINING_JOB_TEMPLATE = ""  # Explicit training-stage RunPod estimate JSON.
if ENDPOINT_CONFIG and TRAINING_JOB_TEMPLATE:
    run_command("sampling-sft-train-job-request", "--config", ENDPOINT_CONFIG, "--template", TRAINING_JOB_TEMPLATE)


## 19. Optional handoff snapshot (CPU)

Select a JSON listing the existing stage plans. Source-only partial reports are valid. This verifies saved evidence and creates an inventory; no model or cloud work starts. See `docs/operations/sampling-sft-handoff.md`.


In [ ]:
HANDOFF_SELECTION = ""  # Explicit existing-stage selection JSON.
HANDOFF_NAME = PLAN_NAME + "-handoff-v1"
if HANDOFF_SELECTION:
    release_arguments = ["--release-commit", SOURCE_RELEASE] if SOURCE_RELEASE else []
    run_command("sampling-sft-handoff-prepare", "--selection", HANDOFF_SELECTION, "--name", HANDOFF_NAME, *release_arguments)
    HANDOFF_CONFIG = ARTIFACT_ROOT / "plans" / HANDOFF_NAME / "sampling-sft.handoff.json"
    run_command("sampling-sft-handoff-build", "--config", HANDOFF_CONFIG)


## 20. Restore and inspect the handoff (CPU)

After reconnecting, select the existing plan and verify it. This checks hashes and status without importing model libraries. A built snapshot may still describe a partial experiment.


In [ ]:
HANDOFF_CONFIG = globals().get("HANDOFF_CONFIG", "")  # Or select a saved sampling-sft.handoff.json.
if HANDOFF_CONFIG:
    run_command("sampling-sft-handoff-verify", "--config", HANDOFF_CONFIG)
    saved_plan = Path(HANDOFF_CONFIG)
    if not saved_plan.is_absolute():
        saved_plan = ARTIFACT_ROOT / saved_plan
    handoff_run = ARTIFACT_ROOT / json.loads(saved_plan.read_text())["run_path"]
    print("Readable report:", handoff_run / "report.md")
    print("Raw inventory:", handoff_run / "results/inventory.json")
